In [ ]:
# Step 1: Install necessary libraries
%pip install transformers datasets torchaudio librosa

In [ ]:
# Step 2: Import libraries
import os
import zipfile
import pandas as pd
from transformers import Wav2Vec2Processor, Wav2Vec2ForCTC
import torch
from tqdm.notebook import tqdm
import librosa

# Step 3: Unzip the dataset
zip_file_path = '/content/drive/MyDrive/RAVDESS(dataset).zip'
unzip_dir = '/content/drive/MyDrive/RAVDESS_dataset/'

# Check if the zip file exists
if not os.path.exists(unzip_dir):
    with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
        zip_ref.extractall(unzip_dir)
        print("Dataset unzipped successfully!")

# Step 4: Load Wav2Vec2 model and processor from Hugging Face
processor = Wav2Vec2Processor.from_pretrained("facebook/wav2vec2-base-960h")
model = Wav2Vec2ForCTC.from_pretrained("facebook/wav2vec2-base-960h")

# Step 5: Process audio files and transcribe them
def transcribe_audio(file_path):
    # Load audio file using librosa
    audio_input, sample_rate = librosa.load(file_path, sr=16000)  # Resample to 16kHz

    # Use Wav2Vec2 processor to process the audio
    input_values = processor(audio_input, return_tensors="pt").input_values

    # Get the logits (raw predictions) from the model
    with torch.no_grad():
        logits = model(input_values).logits

    # Decode the logits to text
    predicted_ids = torch.argmax(logits, dim=-1)
    transcription = processor.batch_decode(predicted_ids)[0]

    return transcription

# Step 6: Prepare CSV to save results
csv_file_path = "/content/drive/MyDrive/transcriptions.csv"
if not os.path.exists(csv_file_path):
    df = pd.DataFrame(columns=["filename", "transcription"])
else:
    df = pd.read_csv(csv_file_path)

# Step 7: Iterate through all audio files and transcribe them
audio_files_dir = unzip_dir

file_list = []
for root, _, files in os.walk(audio_files_dir):  # Iterate through subdirectories to find .wav files
    for file in files:
        if file.endswith('.wav'):
            file_list.append(os.path.join(root, file))  # Store full path

for audio_file_path in tqdm(file_list, desc="Processing Audio Files"):
    transcription = transcribe_audio(audio_file_path)

    # Extract filename from the path
    audio_file = os.path.basename(audio_file_path)

    # Save the result in the dataframe
    df = pd.concat([df, pd.DataFrame({"filename": [audio_file], "transcription": [transcription]})], ignore_index=True)

# Step 8: Save the CSV with the results
df.to_csv(csv_file_path, index=False)
print("Transcriptions saved to CSV file.")

In [ ]:
%pip install PyDub

In [ ]:
from IPython.display import Audio

Audio("/content/drive/MyDrive/RAVDESS_dataset/Actor_01/03-01-01-01-01-01-01.wav")

In [ ]:
from IPython.display import Audio

Audio("/content/drive/MyDrive/RAVDESS_dataset/Actor_01/03-01-01-01-02-01-01.wav")